# Pathway with JuMP: the 2020 to 2050 trajectory

**Open source**: Julia + JuMP + HiGHS. No license needed.
The same tutorial with AMPL is [`02_pathway_ampl.ipynb`](02_pathway_ampl.ipynb).

This notebook runs the EnergyScope pathway model for Quebec with `run_pathway()`, reads the results, draws the charts and
changes the assumptions. It uses the same function and the same results as the AMPL tutorial: you only add `backend="julia"`.

**Before you start:** the Julia backend needs `juliacall` (`pip install juliacall`), which downloads and manages Julia by itself. The first calculation on a machine is slow (installation and compilation); the next ones are fast.

---

## 1. Setup

In [1]:
import sys
sys.path.insert(0, "..")      # the root of the repository, so that `shared` can be imported

from shared.utils import run_pathway

## 2. Basic run

`run_pathway()` computes the whole transition from 2020 to 2050 and returns the results as a plain Python dictionary of pandas
DataFrames. Passing `backend="julia"` selects the open-source model.

> **Time:** about 3 to 8 minutes depending on the machine. Roughly one minute to read the data and build the model, two to four minutes of
> solving, and the rest to compute the result tables. The first run of a session also compiles Julia, which takes a little longer.
>
> **Perfect foresight or myopic:** by default the model solves the whole horizon at once (`N_year_opti=30`). For the rolling-horizon
> ("myopic") mode, add for example `N_year_opti=10, N_year_overlap=5`: the windows are the same as with AMPL.
> With the default limits on the pace of change, the first window of the myopic mode has no feasible solution: section 8 shows how to relax them.
>
> **Solver settings:** by default the solver stops when it is within 0.05 % of the optimum (`mip_gap=5e-4`) and is allowed one hour
> per window (`time_limit=3600`, in seconds). Both can be changed, for example `run_pathway(..., mip_gap=2e-4, time_limit=7200)`.
> A much tighter gap such as `1e-4` can make the solve far longer. If the time limit is reached, the run stops with a message that says so.

In [2]:
results = run_pathway("my_first_run", backend="julia", save_pkl=True)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
julia : window 1/1 OPTIMAL in 175.4 s
[run_pathway] julia : window 1/1 OPTIMAL in 186.1 s
[run_pathway] Window 1/1 done in 207.7s
[run_pathway] Total time: 207.7s


## 3. Exploring results

The dictionary contains about 35 named DataFrames, exactly the ones you get with AMPL. The most commonly used:

In [3]:
# All available result keys
print([k for k in results if results[k] is not None])

['TotalCost', 'C_inv_phase', 'C_inv_phase_no_actu', 'C_inv_phase_tech', 'C_inv_phase_tech_no_actu', 'C_op_phase_tech', 'C_op_phase_res', 'C_op_phase_tech_no_actu', 'C_op_phase_res_no_actu', 'Cost_breakdown', 'Cost_return', 'TotalGwp', 'Resources', 'Assets', 'New_old_decom', 'F_decom', 'Sto_assets', 'Year_balance', 'Transition_cost', 'C_tot_capex', 'C_tot_opex', 'F_Mult_t', 'F_Mult_t_resources', 'F_Mult', 'F_new', 'F_old', 'Annual_Prod', 'Monthly_Prod', 'Sto_levels', 'Number_Of_Units', 'C_inv_phase_CRF', 'tau', 'C_inv_return_phase', 'GwpTransition']


In [6]:
# Installed capacity per technology and year [GW]
results["F_Mult"].loc["YEAR_2030"]

,F_Mult
Technologies,
PV_ROOF,0.009440
TRAFO_ML,20.501174
TRAFO_HM,22.109206
WIND_ONSHORE,3.202800
CCGT,0.333420
...,...
TRUCK_LH_GASOLINE,0.053952
PLANE_FREIGHT,0.128669
BULK_CARRIER,7.466193


In [7]:
# Annual greenhouse-gas emissions per year [kt CO2-eq.]
results["TotalGwp"]

,TotalGWP,Gwp_limit,Gwp_cost
YEAR_2020,54382.705764,inf,-0.0
YEAR_2025,57406.855057,inf,-0.0
YEAR_2030,45481.674354,inf,-0.0
YEAR_2035,37816.000000,37816.0,-0.0
YEAR_2040,30651.039959,37816.0,-0.0
YEAR_2045,27750.317577,37816.0,-0.0
YEAR_2050,0.000000,0.0,-0.0


In [8]:
# Total cost of the transition [M$CAD]
results["Transition_cost"]

,Transition_cost
YEAR_2050,2.190963e+06


In [9]:
# Energy balance of one layer in one year [GWh]: what produces it (+) and what consumes it (-)
results["Year_balance"].loc["YEAR_2030"]["ELECTRICITY_EHV"].dropna()

Elements
TRAFO_EH            -188574.044323
HYDRO_RIVER           73573.195680
NEW_HYDRO_DAM          3401.578086
HYDRO_DAM            127918.533528
TRAIN_ELEC_LD            -4.622797
ELECTRICITY_EHV       33889.000000
END_USES             -14647.640175
ELEC_EXPORT_FIXED    -35556.000000
Name: ELECTRICITY_EHV, dtype: float64

`Year_balance` has one row per technology or resource and one column per layer. A layer is something that is produced and consumed (electricity, natural gas, hydrogen...); positive values are production, negative values are consumption. Replace `"ELECTRICITY_EHV"` or the year with any other name to look at another balance. The complete table would be mostly empty, because a technology only touches a few layers.

## 4. Plotting

The simplest way: add `plot=True` when you run the model. The charts (about 220 HTML files) are generated right after the optimisation, with nothing else to write:



In [10]:
results = run_pathway("my_first_run", backend="julia", save_pkl=True, plot=True)

julia : window 1/1 OPTIMAL in 157.7 s
[run_pathway] julia : window 1/1 OPTIMAL in 157.7 s
[run_pathway] Window 1/1 done in 161.5s
[run_pathway] Total time: 161.5s
Plotting to: c:\Users\matti\Desktop\EnergyScope-Quebec\tutorials\..\projects\opensource\out\my_first_run\graphs
  Plotting [101]: 6b_Monthly_resources_2050.html                         [SKIP] No CO2_STO / STO_CO2 in Monthly_Prod
  Plotting [222]: 18_Elec_monthly_2050.html                                 Dashboard: c:\Users\matti\Desktop\EnergyScope-Quebec\tutorials\..\projects\opensource\out\my_first_run\graphs\index.html

Done — 222 charts saved to c:\Users\matti\Desktop\EnergyScope-Quebec\tutorials\..\projects\opensource\out\my_first_run\graphs


They are written in the output folder of the run, `projects/opensource/out/my_first_run/graphs/`. Open **`0_Summary.html`** in that folder for the overview.

> `plot=True` only acts when the model is actually computed. If you already have results (as in this notebook, where the run of section 2 was computed without it),
> generate the charts afterwards with the cell below. It does exactly the same thing (about a minute).

In [8]:
import io, os, contextlib
sys.path.insert(0, os.path.join("..", "projects", "pathway", "src"))
import plot_results

with contextlib.redirect_stdout(io.StringIO()):          # hide the progress line printed for each chart
    plot_results.run(
        results,
        case_study="my_first_run",
        auto_open=False,
    )
print("Charts written to projects/opensource/out/my_first_run/graphs/ (open 0_Summary.html)")

Charts written to projects/opensource/out/my_first_run/graphs/ (open 0_Summary.html)


## 5. Saving results to disk

Set `save_pkl=True` to write the results to `projects/opensource/out/<case_study>/_Results.pkl`
(the open-source runs have their own `out` folder, apart from the AMPL runs in `projects/pathway/out`).
A line is also added to `projects/opensource/out/_Recap.csv` with the main figures of the run.

With `skip_if_exists=True`, a run that is already saved is not computed again: the saved results are loaded instead.

The run of section 2 was saved (`save_pkl=True`), so the cell below loads it in a few seconds instead of computing it again.

In [9]:
results = run_pathway(
    "my_first_run",
    backend="julia",
    save_pkl=True,
    description="Baseline run with default settings",
    skip_if_exists=True,   # do not recompute if already saved
)

[run_pathway] my_first_run — pkl exists, loading from disk.


## 6. Changing the data with a text file

To test a "what if...?", write your changes in the text file **`my_changes.dat`** (in the folder `example_files` next to this notebook) and pass it with
`extra_files`. It is the same file as in [`03_snapshot_jump_opensource.ipynb`](03_snapshot_jump_opensource.ipynb): ready-to-use examples
(natural gas +50 %, onshore wind -30 %, no nuclear...). Example 1 is switched on; the others are switched off by a `#` at the start of the line. Remove the `#` of an example,
change the number if you wish, and save the file. The standard data files are never modified.

Because the lines are written with `y in YEARS`, they apply to every year of the transition.

> **Time:** the solve time of HiGHS can vary a lot from one run to another: a run with changes can take from a few minutes to about twenty minutes.

In [10]:
results_changes = run_pathway(
    "my_changes_run",
    backend="julia",
    extra_files=["example_files/my_changes.dat"],
    save_pkl=True,
    description="With my_changes.dat",
)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 429210 rows; 392798 cols; 3009394 nonzeros; 7248 integer variables (7248 binary)
Coefficient ranges:
  Matrix  [6e-06, 2e+05]
  Cost    [1e+00, 1e+00]
  Bound   [3e-03, 1e+04]
  RHS     [1e+00, 1e+08]
Presolving model
105645 rows, 63525 cols, 724285 nonzeros 1s
81618 rows, 45170 cols, 603723 nonzeros 2s
71493 rows, 42984 cols, 557381 nonzeros 2s
Presolve reductions: rows 71493(-357717); columns 42984(-349814); nonzeros 557381(-2452013) 

Solving MIP model with:
   71493 rows
   42984 cols (520 binary, 0 integer, 0 implied int., 42464 continuous, 0 domain fixed)
   557381 nonzeros
   Thread count 10 (of 20 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility ju

[run_pathway] Window 1/1 done in 1250.2s


[run_pathway] Total time: 1250.2s


In [11]:
# Compare the total cost of the transition with the baseline run
base = results["Transition_cost"].iloc[-1, 0]
mine = results_changes["Transition_cost"].iloc[-1, 0]
print(f"Baseline: {base:,.0f}   With your changes: {mine:,.0f}   ({(mine - base) / base:+.2%})")

Baseline: 2,190,963   With your changes: 2,202,766   (+0.54%)


> Only plain data files are accepted with Julia (lines such as `let`, `for`, `fix` and `subject to`). A file that declares new variables
> needs the AMPL version.

## 7. GWP budget

Set `gwp_budget=True` to cap the cumulative greenhouse-gas emissions of the whole transition (2020 to 2050) with the built-in default value,
or pass your own number in kt CO2-eq. (for example `gwp_budget=1000000`). The model then has to cut emissions, which usually costs a little more.

> A budget changes the optimisation problem and can make it easier or harder to solve: in our tests the run took from about 2 to 25 minutes depending on the data and the solver settings, and raised the transition cost by a few tenths of a percent. When the budget is binding, the emissions of the solution are equal to it.

In [12]:
results_budget = run_pathway("my_budget_run", backend="julia", gwp_budget=True, save_pkl=True)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 429211 rows; 392798 cols; 3009395 nonzeros; 7248 integer variables (7248 binary)
Coefficient ranges:
  Matrix  [6e-06, 2e+05]
  Cost    [1e+00, 1e+00]
  Bound   [3e-03, 1e+04]
  RHS     [1e+00, 1e+08]
Presolving model
105645 rows, 63525 cols, 724285 nonzeros 1s
81618 rows, 45170 cols, 603723 nonzeros 1s
71493 rows, 42984 cols, 557381 nonzeros 2s
Presolve reductions: rows 71493(-357718); columns 42984(-349814); nonzeros 557381(-2452014) 

Solving MIP model with:
   71493 rows
   42984 cols (520 binary, 0 integer, 0 implied int., 42464 continuous, 0 domain fixed)
   557381 nonzeros
   Thread count 10 (of 20 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility ju

[run_pathway] Window 1/1 done in 1125.1s


[run_pathway] Total time: 1125.1s


In [13]:
# Emissions of the whole transition [kt CO2-eq.], without and with the budget
base_gwp = results["GwpTransition"].iloc[0, 0]
budget_gwp = results_budget["GwpTransition"].iloc[0, 0]
print(f"Baseline: {base_gwp:,.0f}   With the budget: {budget_gwp:,.0f}")

Baseline: 1,131,486   With the budget: 883,428


## 8. Myopic mode (rolling horizon)

In the myopic mode the model optimises the transition window by window (here 10 years per window, 5 years of overlap) instead of
all at once. The default limits on the pace of change and on the cost of each phase are too tight for it: the first window has no
feasible solution. The file [`myopic_limits.dat`](example_files/myopic_limits.dat), relaxes them; it is passed like any other
file of changes.

> **Relaxing the limits changes the problem.** Compare two runs only if they use the same limits: to compare with a perfect-foresight run,
> run that one with `extra_files=["example_files/myopic_limits.dat"]` too.
>
> **Time:** a few minutes.

In [14]:
results_myopic = run_pathway(
    "my_myopic_run",
    backend="julia",
    N_year_opti=10,
    N_year_overlap=5,
    extra_files=["example_files/myopic_limits.dat"],
    save_pkl=True,
    description="Myopic mode, relaxed limits",
)
print("Transition cost of the whole transition:", f"{results_myopic['Transition_cost'].iloc[-1, 0]:,.0f}")

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 181643 rows; 347318 cols; 1164686 nonzeros; 7248 integer variables (7248 binary)
Coefficient ranges:
  Matrix  [6e-06, 2e+05]
  Cost    [1e+00, 1e+00]
  Bound   [3e-03, 1e+04]
  RHS     [1e+00, 1e+08]
Presolving model
26669 rows, 15009 cols, 181100 nonzeros 0s
20031 rows, 9554 cols, 152379 nonzeros 0s
16911 rows, 8882 cols, 129377 nonzeros 0s
Presolve reductions: rows 16911(-164732); columns 8882(-338436); nonzeros 129377(-1035309) 

Solving MIP model with:
   16911 rows
   8882 cols (104 binary, 0 integer, 0 implied int., 8778 continuous, 0 domain fixed)
   129377 nonzeros
   Thread count 10 (of 20 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L 

[run_pathway] Window 1/5 done in 23.9s


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 172189 rows; 353240 cols; 980553 nonzeros; 7248 integer variables (7248 binary)
Coefficient ranges:
  Matrix  [6e-06, 2e+05]
  Cost    [1e+00, 1e+00]
  Bound   [6e-17, 2e+05]
  RHS     [1e+00, 2e+06]
Presolving model
37531 rows, 24259 cols, 246285 nonzeros 0s
28377 rows, 14531 cols, 204447 nonzeros 0s
25213 rows, 13832 cols, 192244 nonzeros 0s
Presolve reductions: rows 25213(-146976); columns 13832(-339408); nonzeros 192244(-788309) 

Solving MIP model with:
   25213 rows
   13832 cols (208 binary, 0 integer, 0 implied int., 13624 continuous, 0 domain fixed)
   192244 nonzeros
   Thread count 10 (of 20 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump;

[run_pathway] Window 2/5 done in 28.6s


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 195752 rows; 360542 cols; 1054197 nonzeros; 7248 integer variables (7248 binary)
Coefficient ranges:
  Matrix  [6e-06, 2e+05]
  Cost    [1e+00, 1e+00]
  Bound   [1e-31, 3e+05]
  RHS     [1e+00, 2e+06]
Presolving model
37685 rows, 25633 cols, 247462 nonzeros 0s
28399 rows, 14635 cols, 204867 nonzeros 0s
25232 rows, 13915 cols, 192675 nonzeros 1s
Presolve reductions: rows 25232(-170520); columns 13915(-346627); nonzeros 192675(-861522) 

Solving MIP model with:
   25232 rows
   13915 cols (208 binary, 0 integer, 0 implied int., 13707 continuous, 0 domain fixed)
   192675 nonzeros
   Thread count 10 (of 20 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump

[run_pathway] Window 3/5 done in 95.6s


... (2 outputs left out)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
MIP has 258373 rows; 390038 cols; 1237587 nonzeros; 7248 integer variables (7248 binary)
Coefficient ranges:
  Matrix  [6e-06, 2e+05]
  Cost    [1e+00, 1e+00]
  Bound   [1e-32, 3e+05]
  RHS     [1e+00, 2e+06]
Presolving model
37899 rows, 28282 cols, 249094 nonzeros 0s
28512 rows, 14913 cols, 205240 nonzeros 0s
25239 rows, 14142 cols, 192599 nonzeros 1s
Presolve reductions: rows 25239(-233134); columns 14142(-375896); nonzeros 192599(-1044988) 

Solving MIP model with:
   25239 rows
   14142 cols (208 binary, 0 integer, 0 implied int., 13934 continuous, 0 domain fixed)
   192599 nonzeros
   Thread count 10 (of 20 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jum

[run_pathway] Window 5/5 done in 33.1s


[run_pathway] Total time: 214.2s


Transition cost of the whole transition: 2,162,741


## 9. Good to know

* The charts and tables are produced by the same code as with AMPL, so you can open a result of either version with the same tools.
* The two versions give the same costs to within the solver tolerance (0.05 % by default, see `mip_gap`). When several combinations of technologies cost almost the same, the details of the operation
  (for example the exact monthly profiles) can differ: compare mainly the **costs** and the **overall trends**.
* Results of open-source runs are in `projects/opensource/out/`, those of AMPL runs in `projects/pathway/out/`.
* One year only? See [`03_snapshot_jump_opensource.ipynb`](03_snapshot_jump_opensource.ipynb).